# 03 · Preparação para a modelagem

**CRISP-DM fase 3 — preparação dos dados** · Turismo e Hotelaria

---

O notebook [`02-ajustes-dados`](02-ajustes-dados.ipynb) entregou a **base tratada**:
119.390 reservas, nenhuma célula vazia, e cada ajuste com motivo e quantidade
registrados em `reports/tables/ajustes-dados.csv`. Ela descreve a reserva em **unidades
de negócio** — dias, diárias, contagens de hóspedes e rótulos de canal — e é
justamente isso que a torna legível para a operação e imprópria para um algoritmo
baseado em distância.

Diferente do tema de mercado imobiliário — onde a maior parte das colunas é numérica
e um pequeno grupo de escalas de qualidade é ordinal —, a base de reservas **não tem
nenhuma coluna ordinal**: é dominada por categorias nominais (tipo de hotel, mês,
regime de refeição, segmento de mercado, tipo de quarto, país...). Por isso este
notebook não carrega um `escala.csv`; ele classifica as colunas em apenas dois grupos,
contínuo/discreto e nominal, e trata à parte a única nominal de altíssima
cardinalidade da base: `country`.

In [ ]:
# ============================================================================
# Importações
# ============================================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# ============================================================================
# Configuração de avisos
# ============================================================================


def warn(*args, **kwargs):
    pass


warnings.warn = warn
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

# ============================================================================
# Escalonamento de variáveis numéricas
# ============================================================================
# MinMaxScaler    → ajusta cada variável (coluna) para o intervalo de 0 a 1.
#
# StandardScaler  → centraliza cada variável (coluna) em média 0 e desvio
#                   padrão 1. Não torna a distribuição normal.
#
# Normalizer      → ajusta cada observação (linha) para norma L2 igual a 1,
#                   ou seja, a soma dos quadrados dos valores da linha é 1.

# ============================================================================
# Codificação de variáveis categóricas
# ============================================================================
# OneHotEncoder   → cria uma coluna binária (0/1) para cada categoria.
#                   Indicado para variáveis nominais.
#
# OrdinalEncoder  → converte cada categoria em um número inteiro. A ordem só
#                   é respeitada se informada em `categories`; caso contrário,
#                   segue a ordem alfabética. Indicado para variáveis ordinais.
#                   Este tema não tem nenhuma coluna ordinal (ver notebook 00).

In [ ]:
# Carregando a base tratada e o dicionário de dados
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

df = pd.read_csv(RAIZ / "data" / "processed" / "hotel_bookings_tratado.csv", sep=";")
dicionario = pd.read_csv(RAIZ / "data" / "processed" / "dicionario.csv", sep=";")

print(f"{df.shape[0]:,} reservas × {df.shape[1]} colunas na base tratada")
print(f"{len(dicionario)} variáveis descritas no dicionário")

## Colunas fora da matriz de modelagem

Seis colunas da base tratada não entram como atributo de entrada do agrupamento:

- **`is_canceled`** é o rótulo. Descreve o desfecho da reserva, não o seu perfil no
  momento em que foi feita — é **reservado** para a avaliação (fase 5), igual ao preço
  reservado no tema de mercado imobiliário.
- **`reservation_status`** e **`reservation_status_date`** são posteriores à reserva
  (vazamento): só existem depois do check-out, cancelamento ou no-show.
- **`arrival_date`** repete, em formato de texto, a mesma informação já presente em
  `arrival_date_year`, `arrival_date_month` e `arrival_date_day_of_month`.
- **`agent`** (334 categorias) e **`company`** (353 categorias) são identificadores de
  altíssima cardinalidade, dominados por uma categoria de ausência (`Sem agência`,
  `Sem empresa`); um One-Hot sobre eles infla a dimensionalidade sem agregar perfil de
  reserva.

In [ ]:
# Removendo colunas que não serão utilizadas no modelo
useless_columns = [
    "is_canceled",
    "reservation_status",
    "reservation_status_date",
    "arrival_date",
    "agent",
    "company",
]

## Reduzindo a cardinalidade de `country`

`country` tem 178 categorias — a maior cardinalidade entre as colunas que seguem para
o modelo. Um One-Hot sobre ela sozinha criaria mais colunas do que todas as outras
nominais somadas, e a cauda tem países com 1 ou 2 reservas, que não sustentam um
perfil. A base é agrupada nos países mais frequentes; o restante vira `Outro`.

In [ ]:
# Agrupando países pouco frequentes em 'Outro'
TOPO_PAISES = 15
frequencia_paises = df["country"].value_counts()
paises_mantidos = frequencia_paises.head(TOPO_PAISES).index
cobertura = frequencia_paises.head(TOPO_PAISES).sum() / len(df)
print(f"Os {TOPO_PAISES} países mais frequentes cobrem {cobertura:.1%} das reservas")

df["country"] = np.where(df["country"].isin(paises_mantidos), df["country"], "Outro")
print(f"Categorias finais em 'country': {df['country'].nunique()}")

In [ ]:
# Seleciona as colunas do dicionário pelo tipo (apenas as presentes na base)
def colunas_do_tipo(*tipos):
    colunas = dicionario.loc[dicionario["Tipo"].isin(tipos), "variavel"]
    return [c for c in colunas if c in df.columns and c not in useless_columns]


# Contínua, discreta e binária (0/1 já codificado — o MinMaxScaler não altera esse intervalo)
continuous_columns = colunas_do_tipo("Contínua", "Discreta", "Binária")

# Ordinal & Nominal — este tema não tem nenhuma coluna ordinal
ordinal_columns = colunas_do_tipo("Ordinal")
nominal_columns = colunas_do_tipo("Nominal")

# As derivações criadas no notebook 02-ajustes-dados
derived_continuous = ["total_nights", "total_guests"]
derived_binary = ["room_changed", "has_special_requests"]
continuous_columns = list(
    dict.fromkeys(continuous_columns + derived_continuous + derived_binary)
)  # sem duplicar

print(
    f"{len(continuous_columns)} contínuas/discretas/binárias · "
    f"{len(ordinal_columns)} ordinais · {len(nominal_columns)} nominais"
)
print("Contínuas/discretas/binárias:", continuous_columns)
print("Nominais:", nominal_columns)

In [ ]:
# Pipelines de pré-processamento para cada tipo de variável
## Contínuas, discretas e binárias
continuous_preprocessor = Pipeline(
    steps=[
        ("normalization", MinMaxScaler())  # Normalização
    ]
)

## Nominais
nominal_preprocessor = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),  # Codificação de variáveis
    ]
)

# Temporais e colunas não listadas são descartadas (remainder='drop')
preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_preprocessor, continuous_columns),
        ("nominal", nominal_preprocessor, nominal_columns),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

In [ ]:
# Definindo features do processamento
X = df.drop(columns=useless_columns, errors="ignore")

# Aplicando o pré-processamento
X_preprocessed = preprocessor.fit_transform(X)
print(f"{X_preprocessed.shape[0]:,} reservas × {X_preprocessed.shape[1]} colunas pré-processadas")

# Salvando a base pré-processada
X_preprocessed.to_csv(
    RAIZ / "data" / "processed" / "hotel_bookings_preprocessed.csv", sep=";", index=False
)
X_preprocessed.head(3)